In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, roc_curve, precision_recall_curve
from imblearn.over_sampling import SMOTE
import copy
import os
import random
import json
from datetime import datetime

In [22]:
### load data
data_path = '/data/jxiao/gas_data/gas_new/values_refine/RGB_diff_refine_label_new_n2.csv'
diff = pd.read_csv(data_path, index_col='id')

X = diff.drop('label', axis=1, inplace=False).to_numpy()
y = diff['label'].to_numpy().astype(int)
print(X.shape)
print('positive samples:', sum(y))

(134, 8)
positive samples: 81


In [29]:
INPUT_DIM = X.shape[1]
EPOCHS = 250
BATCH_SIZE = 20
LR = 2e-4
PATIENCE = 10
N_SPLITS = 5
N_REPEATS = 10  # 重复跑5fold-CV的次数
BASE_SEED = 42
THRESHOLD = 0.5
DROPOUT = 0.3
seed_dir = 'seeds'  # reusable repeat/fold random seeds
result_dir = 'results/4_layers'  # per-run metrics; each run is saved without overwriting old results
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
SEED_FILE = os.path.join(seed_dir, 'repeat_seeds.json') if seed_dir else None
REUSE_SEEDS = True  # True: load existing seeds from SEED_FILE; False: regenerate and overwrite
use_smote = False
use_posweight = False
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def make_unique_dir(parent_dir, name):
    if not parent_dir:
        return None
    os.makedirs(parent_dir, exist_ok=True)
    run_dir = os.path.join(parent_dir, name)
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)
        return run_dir
    suffix = 1
    while True:
        candidate = os.path.join(parent_dir, f"{name}_{suffix}")
        if not os.path.exists(candidate):
            os.makedirs(candidate)
            return candidate
        suffix += 1

def make_unique_file_path(directory, filename):
    path = os.path.join(directory, filename)
    if not os.path.exists(path):
        return path
    stem, ext = os.path.splitext(filename)
    suffix = 1
    while True:
        candidate = os.path.join(directory, f"{stem}_{suffix}{ext}")
        if not os.path.exists(candidate):
            return candidate
        suffix += 1

def build_repeat_seed_plan(n_repeats, n_splits, base_seed):
    seed_entries = []
    for repeat in range(n_repeats):
        repeat_seed = int(base_seed + repeat)
        fold_seeds = [int(repeat_seed * 100 + fold) for fold in range(n_splits)]
        seed_entries.append({
            'repeat': repeat,
            'repeat_seed': repeat_seed,
            'fold_seeds': fold_seeds,
        })
    return {
        'n_repeats': int(n_repeats),
        'n_splits': int(n_splits),
        'base_seed': int(base_seed),
        'seeds': seed_entries,
    }

def save_repeat_seed_plan(seed_plan, seed_file):
    if seed_file is None:
        return
    seed_dir = os.path.dirname(seed_file)
    if seed_dir:
        os.makedirs(seed_dir, exist_ok=True)
    with open(seed_file, 'w', encoding='utf-8') as f:
        json.dump(seed_plan, f, indent=2)

def load_repeat_seed_plan(seed_file, n_repeats=None, n_splits=None):
    if seed_file is None:
        raise ValueError("seed_file is None; set seed_dir/SEED_FILE before loading reusable seeds")
    with open(seed_file, 'r', encoding='utf-8') as f:
        seed_plan = json.load(f)
    if n_repeats is not None and seed_plan['n_repeats'] != n_repeats:
        raise ValueError(f"Seed file n_repeats={seed_plan['n_repeats']} does not match N_REPEATS={n_repeats}")
    if n_splits is not None and seed_plan['n_splits'] != n_splits:
        raise ValueError(f"Seed file n_splits={seed_plan['n_splits']} does not match N_SPLITS={n_splits}")
    return seed_plan

def prepare_repeat_seed_plan(seed_file, n_repeats, n_splits, base_seed, reuse=False):
    if seed_file is None:
        print("SEED_FILE is None; using in-memory repeat seeds only")
        return build_repeat_seed_plan(n_repeats, n_splits, base_seed)
    if reuse and os.path.exists(seed_file):
        seed_plan = load_repeat_seed_plan(seed_file, n_repeats=n_repeats, n_splits=n_splits)
        print(f"Loaded repeat seeds from {seed_file}")
    else:
        seed_plan = build_repeat_seed_plan(n_repeats, n_splits, base_seed)
        save_repeat_seed_plan(seed_plan, seed_file)
        print(f"Saved repeat seeds to {seed_file}")
    return seed_plan

def get_repeat_seed(seed_plan, repeat):
    return int(seed_plan['seeds'][repeat]['repeat_seed'])

def get_fold_seed(seed_plan, repeat, fold):
    return int(seed_plan['seeds'][repeat]['fold_seeds'][fold])

def write_run_config(config_path, config):
    with open(config_path, 'w', encoding='utf-8') as f:
        for key, value in config.items():
            f.write(f"{key}: {value}\n")

def save_repeat_average_results(repeat_results, run_dir, seed_file, config):
    if not run_dir:
        return None
    result_df = pd.DataFrame(repeat_results)
    if 'fold_seeds' in result_df.columns:
        result_df['fold_seeds'] = result_df['fold_seeds'].apply(json.dumps)

    metric_names = ['acc', 'f1', 'auc', 'precision', 'recall']
    summary = {
        'seed_file': seed_file,
        'n_repeats': len(repeat_results),
        'n_splits': config['N_SPLITS'],
        'metrics': {
            metric: {
                'mean': float(result_df[metric].mean()),
                'std': float(result_df[metric].std(ddof=0)),
            }
            for metric in metric_names
        },
        'config': config,
    }

    csv_path = make_unique_file_path(run_dir, 'repeat_average_results.csv')
    json_path = make_unique_file_path(run_dir, 'repeat_average_summary.json')
    result_df.to_csv(csv_path, index=False)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)
    print(f"Saved repeat average results to {csv_path}")
    print(f"Saved repeat average summary to {json_path}")
    return {'repeat_average_csv': csv_path, 'summary_json': json_path}

RUN_DIR = make_unique_dir(result_dir, RUN_ID)
run_config = {
    'run_id': RUN_ID,
    'run_dir': RUN_DIR,
    'data_path': data_path,
    'INPUT_DIM': INPUT_DIM,
    'EPOCHS': EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'LR': LR,
    'PATIENCE': PATIENCE,
    'N_SPLITS': N_SPLITS,
    'N_REPEATS': N_REPEATS,
    'BASE_SEED': BASE_SEED,
    'seed_dir': seed_dir,
    'result_dir': result_dir,
    'SEED_FILE': SEED_FILE,
    'REUSE_SEEDS': REUSE_SEEDS,
    'THRESHOLD': THRESHOLD,
    'DROPOUT': DROPOUT,
    'use_smote': use_smote,
    'use_posweight': use_posweight,
}

if RUN_DIR:
    write_run_config(os.path.join(RUN_DIR, 'config.txt'), run_config)
    print(f"Run artifacts will be saved to {RUN_DIR}")

class SmallMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # self.net = nn.Sequential(
        #     nn.Linear(input_dim, 64),  # 64
        #     nn.ReLU(),
        #     nn.Dropout(DROPOUT),  # 0.3
        #     nn.Linear(64, 32),  # 32
        #     nn.ReLU(),
        #     nn.Dropout(DROPOUT),  # 0.3
        #     nn.Linear(32, 1)
        # )
        # self.net = nn.Sequential(
        #     nn.Linear(input_dim, 128), 
        #     nn.ReLU(),
        #     nn.Dropout(DROPOUT), 
        #     nn.Linear(128, 64), 
        #     nn.ReLU(),
        #     nn.Dropout(DROPOUT),
        #     nn.Linear(64, 32), 
        #     nn.ReLU(),
        #     nn.Dropout(DROPOUT), 
        #     nn.Linear(32, 1)
        # )
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), 
            nn.ReLU(),
            nn.Dropout(DROPOUT), 
            nn.Linear(256, 128), 
            nn.ReLU(),
            nn.Dropout(DROPOUT), 
            nn.Linear(128, 64), 
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(64, 32), 
            nn.ReLU(),
            nn.Dropout(DROPOUT), 
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

Run artifacts will be saved to results/4_layers/20260518_161013


In [30]:
all_acc, all_f1, all_auc, all_precision, all_recall = [], [], [], [], [] # eval metrics
all_fpr, all_tpr = [], []  # ROC curves
all_precision_thre, all_recall_thre, all_thresholds = [], [], []  # Precision / Recall vs. Threshold curves
y_val_list, y_pred_list, y_label_list, prob_list = [], [], [], []  # save ground truth and predictions
repeat_results = []
acc_per_epoch = [[[] for _ in range(N_SPLITS)] for _ in range(N_REPEATS)] # model performance evolution curves
f1_per_epoch = [[[] for _ in range(N_SPLITS)] for _ in range(N_REPEATS)]
auc_per_epoch = [[[] for _ in range(N_SPLITS)] for _ in range(N_REPEATS)]
train_loss_list = [[[] for _ in range(N_SPLITS)] for _ in range(N_REPEATS)] # loss curves
val_loss_list = [[[] for _ in range(N_SPLITS)] for _ in range(N_REPEATS)]
seed_plan = prepare_repeat_seed_plan(SEED_FILE, N_REPEATS, N_SPLITS, BASE_SEED, reuse=REUSE_SEEDS)

for repeat in range(N_REPEATS):
    repeat_seed = get_repeat_seed(seed_plan, repeat)
    set_seed(repeat_seed)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=repeat_seed)
    repeat_acc, repeat_f1, repeat_auc, repeat_precision, repeat_recall = [], [], [], [], []

    print(f"\n===== Repeat {repeat + 1}/{N_REPEATS} =====")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        fold_seed = get_fold_seed(seed_plan, repeat, fold)
        set_seed(fold_seed)

        print(f"\n===== Repeat {repeat + 1} Fold {fold + 1} =====")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # --- normalization ---
        scaler = StandardScaler()
        scaler.fit(X_train)

        X_train = scaler.transform(X_train)
        X_val = scaler.transform(X_val)

        # --- SMOTE ---
        if use_smote:
            smote = SMOTE(random_state=fold_seed)
            X_train, y_train = smote.fit_resample(X_train, y_train)

        X_train = torch.tensor(X_train, dtype=torch.float32)
        y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
        X_val = torch.tensor(X_val, dtype=torch.float32)
        y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
        y_val_np = y_val.squeeze().cpu().numpy().astype(int)

        # --- DataLoader ---
        generator = torch.Generator()
        generator.manual_seed(fold_seed)
        train_loader = DataLoader(
            TensorDataset(X_train, y_train),
            batch_size=BATCH_SIZE,
            shuffle=True,
            generator=generator,
        )
        val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)

        # --- Model & Optimizer ---
        model = SmallMLP(INPUT_DIM).to(device)
        if use_posweight:
            num_pos = y_train.sum()
            num_neg = y_train.shape[0] - num_pos
            # pos_weight = torch.tensor([num_neg / num_pos]).to(device)  # 正例的权重
            pos_weight = torch.tensor([3.0]).to(device)  # 手动设定正例权重
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        else:
            criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
        # optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4)

        # --- Train ---
        best_val_loss = float('inf')
        best_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0

        for epoch in range(EPOCHS):
            model.train()
            train_loss = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                preds = model(xb)
                loss = criterion(preds, yb)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * xb.size(0)
            train_loss /= len(train_loader.dataset)

            #--- Validation ---
            model.eval()
            val_loss = 0.0
            all_preds, all_labels = [], []
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    preds = model(xb)
                    loss = criterion(preds, yb)
                    val_loss += loss.item() * xb.size(0)
                    all_preds.append(torch.sigmoid(preds).cpu().numpy())
                    all_labels.append(yb.cpu().numpy())
            val_loss /= len(val_loader.dataset)

            all_preds = np.vstack(all_preds)
            all_labels = np.vstack(all_labels)
            pred_labels = (all_preds > 0.5).astype(int)

            acc = accuracy_score(all_labels, pred_labels)
            f1 = f1_score(all_labels, pred_labels, zero_division=0)
            auc = roc_auc_score(all_labels, all_preds)

            acc_per_epoch[repeat][fold].append(acc)
            f1_per_epoch[repeat][fold].append(f1)
            auc_per_epoch[repeat][fold].append(auc)

            train_loss_list[repeat][fold].append(train_loss)
            val_loss_list[repeat][fold].append(val_loss)

            #--- Early stopping ---
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_weights = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE or epoch == EPOCHS - 1:
                    print("Stop in epoch {}!".format(epoch + 1))
                    break
        model.load_state_dict(best_weights)

        # --- Final evaluation ---
        model.eval()
        with torch.no_grad():
            preds = torch.sigmoid(model(X_val.to(device))).cpu().numpy()
            pred_labels = (preds > THRESHOLD).astype(int)
        prob_list.append(preds)

        acc = accuracy_score(y_val_np, pred_labels)
        f1 = f1_score(y_val_np, pred_labels, zero_division=0)
        auc = roc_auc_score(y_val_np, preds)
        precision = precision_score(y_val_np, pred_labels, zero_division=0)
        recall = recall_score(y_val_np, pred_labels, zero_division=0)

        fpr, tpr, _ = roc_curve(y_val_np, preds)
        precision_thre, recall_thre, thresholds = precision_recall_curve(y_val_np, preds)

        print(f"Repeat {repeat + 1} Fold {fold + 1} val results: ACC={acc:.3f}, F1={f1:.3f}, precision={precision:.3f}, recall={recall:.3f}")
        all_acc.append(acc)
        all_f1.append(f1)
        all_auc.append(auc)
        all_precision.append(precision)
        all_recall.append(recall)

        repeat_acc.append(acc)
        repeat_f1.append(f1)
        repeat_auc.append(auc)
        repeat_precision.append(precision)
        repeat_recall.append(recall)

        all_fpr.append(fpr)
        all_tpr.append(tpr)
        all_precision_thre.append(precision_thre)
        all_recall_thre.append(recall_thre)
        all_thresholds.append(thresholds)

        y_val_list.append(y_val_np.tolist())
        y_pred_list.append(preds.squeeze().tolist())
        y_label_list.append(pred_labels.squeeze().tolist())

    repeat_results.append({
        'repeat': repeat + 1,
        'repeat_seed': repeat_seed,
        'fold_seeds': seed_plan['seeds'][repeat]['fold_seeds'],
        'acc': np.mean(repeat_acc),
        'f1': np.mean(repeat_f1),
        'auc': np.mean(repeat_auc),
        'precision': np.mean(repeat_precision),
        'recall': np.mean(repeat_recall),
    })
    print(f"\n===== Repeat {repeat + 1} Results ({N_SPLITS}-Fold Avg) =====")
    print(f"ACC={np.mean(repeat_acc):.3f} ± {np.std(repeat_acc):.3f}")
    print(f"F1 ={np.mean(repeat_f1):.3f} ± {np.std(repeat_f1):.3f}")
    print(f"AUC={np.mean(repeat_auc):.3f} ± {np.std(repeat_auc):.3f}")
    print(f"Precision={np.mean(repeat_precision):.3f} ± {np.std(repeat_precision):.3f}")
    print(f"Recall={np.mean(repeat_recall):.3f} ± {np.std(repeat_recall):.3f}")

result_paths = save_repeat_average_results(repeat_results, RUN_DIR, SEED_FILE, run_config)

Loaded repeat seeds from seeds/repeat_seeds.json

===== Repeat 1/10 =====

===== Repeat 1 Fold 1 =====
Stop in epoch 55!
Repeat 1 Fold 1 val results: ACC=0.926, F1=0.941, precision=0.941, recall=0.941

===== Repeat 1 Fold 2 =====
Stop in epoch 39!
Repeat 1 Fold 2 val results: ACC=0.741, F1=0.800, precision=0.737, recall=0.875

===== Repeat 1 Fold 3 =====
Stop in epoch 52!
Repeat 1 Fold 3 val results: ACC=0.815, F1=0.848, precision=0.824, recall=0.875

===== Repeat 1 Fold 4 =====
Stop in epoch 43!
Repeat 1 Fold 4 val results: ACC=0.778, F1=0.812, precision=0.812, recall=0.812

===== Repeat 1 Fold 5 =====
Stop in epoch 46!
Repeat 1 Fold 5 val results: ACC=0.846, F1=0.882, precision=0.833, recall=0.938

===== Repeat 1 Results (5-Fold Avg) =====
ACC=0.821 ± 0.063
F1 =0.857 ± 0.051
AUC=0.861 ± 0.041
Precision=0.829 ± 0.065
Recall=0.888 ± 0.048

===== Repeat 2/10 =====

===== Repeat 2 Fold 1 =====
Stop in epoch 50!
Repeat 2 Fold 1 val results: ACC=0.815, F1=0.857, precision=0.833, recall=0.8

In [25]:
repeat_acc = [r['acc'] for r in repeat_results]
repeat_f1 = [r['f1'] for r in repeat_results]
repeat_auc = [r['auc'] for r in repeat_results]
repeat_precision = [r['precision'] for r in repeat_results]
repeat_recall = [r['recall'] for r in repeat_results]

print(f"\n===== Overall Results ({N_REPEATS} Repeats of {N_SPLITS}-Fold Avg) =====")
print(f"ACC={np.mean(repeat_acc):.3f} ± {np.std(repeat_acc):.3f}")
print(f"F1 ={np.mean(repeat_f1):.3f} ± {np.std(repeat_f1):.3f}")
print(f"AUC={np.mean(repeat_auc):.3f} ± {np.std(repeat_auc):.3f}")
print(f"Precision={np.mean(repeat_precision):.3f} ± {np.std(repeat_precision):.3f}")
print(f"Recall={np.mean(repeat_recall):.3f} ± {np.std(repeat_recall):.3f}")


===== Overall Results (10 Repeats of 5-Fold Avg) =====
ACC=0.801 ± 0.019
F1 =0.845 ± 0.013
AUC=0.856 ± 0.012
Precision=0.804 ± 0.025
Recall=0.896 ± 0.010


In [28]:
repeat_acc = [r['acc'] for r in repeat_results]
repeat_f1 = [r['f1'] for r in repeat_results]
repeat_auc = [r['auc'] for r in repeat_results]
repeat_precision = [r['precision'] for r in repeat_results]
repeat_recall = [r['recall'] for r in repeat_results]

print(f"\n===== Overall Results ({N_REPEATS} Repeats of {N_SPLITS}-Fold Avg of 3 hidden layers) =====")
print(f"ACC={np.mean(repeat_acc):.3f} ± {np.std(repeat_acc):.3f}")
print(f"F1 ={np.mean(repeat_f1):.3f} ± {np.std(repeat_f1):.3f}")
print(f"AUC={np.mean(repeat_auc):.3f} ± {np.std(repeat_auc):.3f}")
print(f"Precision={np.mean(repeat_precision):.3f} ± {np.std(repeat_precision):.3f}")
print(f"Recall={np.mean(repeat_recall):.3f} ± {np.std(repeat_recall):.3f}")


===== Overall Results (10 Repeats of 5-Fold Avg of 3 hidden layers) =====
ACC=0.802 ± 0.009
F1 =0.845 ± 0.007
AUC=0.856 ± 0.013
Precision=0.806 ± 0.011
Recall=0.896 ± 0.010


In [31]:
repeat_acc = [r['acc'] for r in repeat_results]
repeat_f1 = [r['f1'] for r in repeat_results]
repeat_auc = [r['auc'] for r in repeat_results]
repeat_precision = [r['precision'] for r in repeat_results]
repeat_recall = [r['recall'] for r in repeat_results]

print(f"\n===== Overall Results ({N_REPEATS} Repeats of {N_SPLITS}-Fold Avg of 4 hidden layers) =====")
print(f"ACC={np.mean(repeat_acc):.3f} ± {np.std(repeat_acc):.3f}")
print(f"F1 ={np.mean(repeat_f1):.3f} ± {np.std(repeat_f1):.3f}")
print(f"AUC={np.mean(repeat_auc):.3f} ± {np.std(repeat_auc):.3f}")
print(f"Precision={np.mean(repeat_precision):.3f} ± {np.std(repeat_precision):.3f}")
print(f"Recall={np.mean(repeat_recall):.3f} ± {np.std(repeat_recall):.3f}")


===== Overall Results (10 Repeats of 5-Fold Avg of 4 hidden layers) =====
ACC=0.798 ± 0.013
F1 =0.843 ± 0.009
AUC=0.859 ± 0.008
Precision=0.803 ± 0.017
Recall=0.895 ± 0.015


In [ ]:
import pandas as pd
from scipy.stats import ttest_rel

df2 = pd.read_csv('/data/jxiao/code/mywork/gas_sensor/rebuttle/results/2_layers/20260518_120928/repeat_average_results.csv')
df3 = pd.read_csv('/data/jxiao/code/mywork/gas_sensor/rebuttle/results/3_layers/20260518_122102/repeat_average_results.csv')
df4 = pd.read_csv('/data/jxiao/code/mywork/gas_sensor/rebuttle/results/4_layers/20260518_161013/repeat_average_results.csv')

ori_acc = df2.acc
t_stat1, p_value1 = ttest_rel(ori_acc, df3.acc)
t_stat2, p_value2 = ttest_rel(ori_acc, df4.acc)
diff1 = df3.acc - ori_acc
diff2 = df4.acc - ori_acc
print(f"3 vs 2 layers: t-statistic={t_stat1:.4f}, p-value={p_value1:.4f}, mean diff={diff1.mean():.4f}")
print(f"4 vs 2 layers: t-statistic={t_stat2:.4f}, p-value={p_value2:.4f}, mean diff={diff2.mean():.4f}")

2 vs 3 layers: t-statistic=-0.0907, p-value=0.9297, mean diff=0.0006
2 vs 4 layers: t-statistic=0.4167, p-value=0.6867, mean diff=-0.0031
